In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# I-monitor o i-cancel ang isang job

Tingnan ang listahan ng iyong mga workload sa [Workloads page](https://quantum.cloud.ibm.com/workloads).

## Tingnan ang status ng job
Pumunta sa iyong [Workloads table](https://quantum.cloud.ibm.com/workloads) at tingnan ang Status column para malaman kung natapos na o nabigo ang isang job.

## Tingnan ang natitirang paggamit
Pumunta sa iyong [Instances table](https://quantum.cloud.ibm.com/instances) at piliin ang tab na nauugnay sa plano na gusto mong tingnan ang natitirang paggamit. Ipinapakita ang kabuuang oras na ginamit at kabuuang oras na natitira sa iyong plano.

## Tingnan ang mga sukatan sa bilang ng mga job at workload na na-submit
Pumunta sa [Analytics page](https://quantum.cloud.ibm.com/analytics) para makita ang kabuuang bilang ng mga job na na-submit, kasama na ang bilang ng mga batch workload at session workload. Tandaan na makikita mo lang ang Analytics page para sa mga account na pag-aari mo o pinamamahalaan mo.

## I-monitor ang isang job
Gamitin ang job instance para suriin ang status ng job o kunin ang mga resulta sa pamamagitan ng pagtawag ng naaangkop na command:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Tingnan ang mga resulta ng job kaagad pagkatapos nitong matapos. Available ang mga resulta ng job pagkatapos matapos ang job. Kaya naman, ang job.result() ay isang blocking call hanggang sa matapos ang job. |
| job.job\_id()                 | Ibalik ang ID na natatanging nagtatukoy sa job na iyon. Kailangan ang job ID para makuha ang mga resulta ng job sa ibang pagkakataon. Kaya naman, inirerekomenda na i-save mo ang mga ID ng mga job na maaaring gusto mong kunin sa ibang pagkakataon. |
| job.status()                  | Suriin ang status ng job.                                                                                                                                                                                    |
| job = service.job(\<job\_id>) | Kunin ang isang job na na-submit mo noon. Kailangan ng job ID para sa call na ito.                                                                                                                           |

<span id="retrieve-later"></span>
## Kunin ang mga resulta ng job sa ibang pagkakataon
Tumawag ng `service.job(\<job\_id>)` para makuha ang isang job na na-submit mo noon. Kung wala kang job ID, o kung gusto mong makuha ang maraming job nang sabay-sabay — kasama na ang mga job mula sa mga retired na QPU (quantum processing units) — tumawag na lang ng `service.jobs()` na may optional na mga filter. Tingnan ang [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** Ibinabalik din ng `service.jobs()` ang mga job na pinatakbo mula sa deprecated na `qiskit-ibm-provider` package. Hindi na available ang mga job na na-submit ng mas lumang (at deprecated na rin) `qiskit-ibmq-provider` package.

### Halimbawa
Ibinabalik ng halimbawang ito ang 10 pinakabagong runtime job na pinatakbo sa `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>